### Setup & load driver-race dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

race = pd.read_csv(PROCESSED / "driver_race_dataset.csv")

print("Loaded driver-race dataset:", race.shape)
race.head()

Loaded driver-race dataset: (6911, 46)


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,teammate_qualifying,teammate_grid,teammate_points,beat_teammate,finish_gap,beat_teammate_quali,qualifying_gap,beat_teammate_grid,grid_gap,points_gap
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,2.0,2.0,18.0,1.0,1.0,0.0,-1.0,0.0,-1.0,7.0
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,3.0,3.0,25.0,0.0,-1.0,1.0,1.0,1.0,1.0,-7.0
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,hamilton,...,8.0,8.0,6.0,1.0,4.0,1.0,4.0,1.0,4.0,9.0
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,vettel,...,6.0,6.0,4.0,1.0,4.0,1.0,5.0,1.0,5.0,8.0
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,rosberg,...,7.0,7.0,8.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0


### Add season context

In [2]:
race = race.sort_values(["season", "round", "constructor_id", "driver_id"]).reset_index(drop=True)

# Season progress helps model car development through the year
season_rounds = (
    race[["season", "round"]]
    .drop_duplicates()
    .groupby("season")["round"]
    .max()
    .rename("season_total_rounds")
    .reset_index()
)

race = race.merge(season_rounds, on="season", how="left")

race["season_progress"] = race["round"] / race["season_total_rounds"]

def get_season_phase(progress):
    if progress <= 1/3:
        return "early"
    elif progress <= 2/3:
        return "middle"
    else:
        return "late"

race["season_phase"] = race["season_progress"].apply(get_season_phase)

def get_regulation_era(season):
    if season <= 2013:
        return "2010_2013_v8"
    elif season <= 2021:
        return "2014_2021_hybrid"
    else:
        return "2022_present_ground_effect"

race["regulation_era"] = race["season"].apply(get_regulation_era)

race[["season", "round", "season_total_rounds", "season_progress", "season_phase", "regulation_era"]].drop_duplicates().head()

,season,round,season_total_rounds,season_progress,season_phase,regulation_era
0,2010,1,19,0.052632,early,2010_2013_v8
24,2010,2,19,0.105263,early,2010_2013_v8
48,2010,3,19,0.157895,early,2010_2013_v8
72,2010,4,19,0.210526,early,2010_2013_v8
96,2010,5,19,0.263158,early,2010_2013_v8


### Build constructor-race table

In [3]:
constructor = (
    race
    .groupby([
        "season",
        "round",
        "race_id",
        "constructor_id",
        "constructor_name",
        "constructor_race_id",
        "season_total_rounds",
        "season_progress",
        "season_phase",
        "regulation_era"
    ], as_index=False)
    .agg(
        constructor_entry_count=("driver_id", "count"),
        constructor_avg_finish=("finish_position", "mean"),
        constructor_finish_std=("finish_position", "std"),
        constructor_avg_grid=("grid_position", "mean"),
        constructor_grid_std=("grid_position", "std"),
        constructor_avg_qualifying=("qualifying_position", "mean"),
        constructor_qualifying_std=("qualifying_position", "std"),
        constructor_total_points=("points", "sum"),
        constructor_avg_points=("points", "mean"),
        constructor_avg_positions_gained=("positions_gained", "mean"),
        constructor_finished_rate=("finished", "mean"),
        constructor_mechanical_dnf_rate=("mechanical_dnf", "mean"),
        constructor_crash_dnf_rate=("crash_dnf", "mean"),
        constructor_other_dnf_rate=("other_dnf", "mean")
    )
)

# Single-car entries are valid but need to be flagged
constructor["single_car_entry"] = (constructor["constructor_entry_count"] == 1).astype(int)

# Std is undefined for single-car entries, so set to 0
std_cols = [
    "constructor_finish_std",
    "constructor_grid_std",
    "constructor_qualifying_std"
]

constructor[std_cols] = constructor[std_cols].fillna(0)

print("Constructor-race table:", constructor.shape)
print("Duplicate constructor-race rows:", constructor.duplicated(["season", "round", "constructor_id"]).sum())

constructor.head()

Constructor-race table: (3457, 25)
Duplicate constructor-race rows: 0


,season,round,race_id,constructor_id,constructor_name,constructor_race_id,season_total_rounds,season_progress,season_phase,regulation_era,...,constructor_avg_qualifying,constructor_qualifying_std,constructor_total_points,constructor_avg_points,constructor_avg_positions_gained,constructor_finished_rate,constructor_mechanical_dnf_rate,constructor_crash_dnf_rate,constructor_other_dnf_rate,single_car_entry
0,2010,1,2010_R1,ferrari,Ferrari,2010_R1_ferrari,19,0.052632,early,2010_2013_v8,...,2.5,0.707107,43.0,21.5,1.0,1.0,0.0,0.0,0.0,0
1,2010,1,2010_R1,force_india,Force India,2010_R1_force_india,19,0.052632,early,2010_2013_v8,...,11.0,1.414214,2.0,1.0,0.5,1.0,0.0,0.0,0.0,0
2,2010,1,2010_R1,hrt,HRT,2010_R1_hrt,19,0.052632,early,2010_2013_v8,...,23.5,0.707107,0.0,0.0,2.0,0.0,0.5,0.5,0.0,0
3,2010,1,2010_R1,lotus_racing,Lotus,2010_R1_lotus_racing,19,0.052632,early,2010_2013_v8,...,20.5,0.707107,0.0,0.0,4.5,0.5,0.5,0.0,0.0,0
4,2010,1,2010_R1,mclaren,McLaren,2010_R1_mclaren,19,0.052632,early,2010_2013_v8,...,6.0,2.828427,21.0,10.5,1.0,1.0,0.0,0.0,0.0,0


### Create prior-only constructor features

In [4]:
constructor = constructor.sort_values(
    ["constructor_id", "season", "round"]
).reset_index(drop=True)

base_feature_cols = [
    "constructor_avg_finish",
    "constructor_finish_std",
    "constructor_avg_grid",
    "constructor_grid_std",
    "constructor_avg_qualifying",
    "constructor_qualifying_std",
    "constructor_total_points",
    "constructor_avg_points",
    "constructor_avg_positions_gained",
    "constructor_finished_rate",
    "constructor_mechanical_dnf_rate",
    "constructor_crash_dnf_rate",
    "constructor_other_dnf_rate"
]

for col in base_feature_cols:
    # Prior-only expanding average prevents current-race leakage
    constructor[f"{col}_expanding_prior"] = (
        constructor
        .groupby("constructor_id")[col]
        .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )

    # Prior-only recent form over last 5 races
    constructor[f"{col}_last5_prior"] = (
        constructor
        .groupby("constructor_id")[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    )

    # Prior-only short-term form over last 3 races
    constructor[f"{col}_last3_prior"] = (
        constructor
        .groupby("constructor_id")[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )

constructor["constructor_has_prior_history"] = (
    constructor["constructor_avg_finish_expanding_prior"].notna()
).astype(int)

print("Prior history counts:")
print(constructor["constructor_has_prior_history"].value_counts())

constructor.head()

Prior history counts:
constructor_has_prior_history
1    3434
0      23
Name: count, dtype: int64


,season,round,race_id,constructor_id,constructor_name,constructor_race_id,season_total_rounds,season_progress,season_phase,regulation_era,...,constructor_mechanical_dnf_rate_expanding_prior,constructor_mechanical_dnf_rate_last5_prior,constructor_mechanical_dnf_rate_last3_prior,constructor_crash_dnf_rate_expanding_prior,constructor_crash_dnf_rate_last5_prior,constructor_crash_dnf_rate_last3_prior,constructor_other_dnf_rate_expanding_prior,constructor_other_dnf_rate_last5_prior,constructor_other_dnf_rate_last3_prior,constructor_has_prior_history
0,2019,1,2019_R1,alfa,Alfa Romeo,2019_R1_alfa,21,0.047619,early,2014_2021_hybrid,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2019,2,2019_R2,alfa,Alfa Romeo,2019_R2_alfa,21,0.095238,early,2014_2021_hybrid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2019,3,2019_R3,alfa,Alfa Romeo,2019_R3_alfa,21,0.142857,early,2014_2021_hybrid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,2019,4,2019_R4,alfa,Alfa Romeo,2019_R4_alfa,21,0.190476,early,2014_2021_hybrid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,2019,5,2019_R5,alfa,Alfa Romeo,2019_R5_alfa,21,0.238095,early,2014_2021_hybrid,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


### Create development and momentum features

In [5]:
# Positive finish/grid/quali trend means recent form is better than long-term form
constructor["constructor_finish_trend_last3_vs_expanding"] = (
    constructor["constructor_avg_finish_expanding_prior"]
    - constructor["constructor_avg_finish_last3_prior"]
)

constructor["constructor_grid_trend_last3_vs_expanding"] = (
    constructor["constructor_avg_grid_expanding_prior"]
    - constructor["constructor_avg_grid_last3_prior"]
)

constructor["constructor_qualifying_trend_last3_vs_expanding"] = (
    constructor["constructor_avg_qualifying_expanding_prior"]
    - constructor["constructor_avg_qualifying_last3_prior"]
)

# Positive points trend means recent scoring is better than long-term scoring
constructor["constructor_points_trend_last3_vs_expanding"] = (
    constructor["constructor_total_points_last3_prior"]
    - constructor["constructor_total_points_expanding_prior"]
)

# Momentum compares very recent form to recent form
constructor["constructor_finish_momentum_last3_vs_last5"] = (
    constructor["constructor_avg_finish_last5_prior"]
    - constructor["constructor_avg_finish_last3_prior"]
)

constructor["constructor_points_momentum_last3_vs_last5"] = (
    constructor["constructor_total_points_last3_prior"]
    - constructor["constructor_total_points_last5_prior"]
)

constructor[[
    "season", "round", "constructor_name",
    "constructor_finish_trend_last3_vs_expanding",
    "constructor_points_trend_last3_vs_expanding",
    "constructor_finish_momentum_last3_vs_last5",
    "constructor_points_momentum_last3_vs_last5"
]].head(20)

,season,round,constructor_name,constructor_finish_trend_last3_vs_expanding,constructor_points_trend_last3_vs_expanding,constructor_finish_momentum_last3_vs_last5,constructor_points_momentum_last3_vs_last5
0,2019,1,Alfa Romeo,NaN,NaN,NaN,NaN
1,2019,2,Alfa Romeo,0.000000,0.000000,0.000000,0.000000
2,2019,3,Alfa Romeo,0.000000,0.000000,0.000000,0.000000
3,2019,4,Alfa Romeo,0.000000,0.000000,0.000000,0.000000
4,2019,5,Alfa Romeo,0.208333,-0.250000,0.208333,-0.250000
5,2019,6,Alfa Romeo,-0.966667,-1.600000,-0.966667,-1.600000
6,2019,7,Alfa Romeo,-1.916667,-1.833333,-1.666667,-1.466667
7,2019,8,Alfa Romeo,-2.738095,-1.857143,-1.666667,-0.600000
8,2019,9,Alfa Romeo,-1.750000,-0.375000,-0.600000,0.600000
9,2019,10,Alfa Romeo,0.722222,0.555556,1.933333,1.200000


### Create car performance score

In [6]:
def z_score(series):
    std = series.std(skipna=True)
    if std == 0 or pd.isna(std):
        return series * np.nan
    return (series - series.mean(skipna=True)) / std

score_components = pd.DataFrame(index=constructor.index)

# Lower finish/grid/quali is better
score_components["finish_component"] = z_score(-constructor["constructor_avg_finish_last5_prior"])
score_components["grid_component"] = z_score(-constructor["constructor_avg_grid_last5_prior"])
score_components["qualifying_component"] = z_score(-constructor["constructor_avg_qualifying_last5_prior"])

# Higher points and reliability are better
score_components["points_component"] = z_score(constructor["constructor_total_points_last5_prior"])
score_components["reliability_component"] = z_score(constructor["constructor_finished_rate_last5_prior"])

# Lower mechanical DNF rate is better
score_components["mechanical_component"] = z_score(-constructor["constructor_mechanical_dnf_rate_last5_prior"])

constructor["car_performance_score_raw"] = score_components.mean(axis=1, skipna=True)

score_min = constructor.loc[
    constructor["constructor_has_prior_history"] == 1,
    "car_performance_score_raw"
].min()

score_max = constructor.loc[
    constructor["constructor_has_prior_history"] == 1,
    "car_performance_score_raw"
].max()

constructor["car_performance_score"] = (
    100
    * (constructor["car_performance_score_raw"] - score_min)
    / (score_max - score_min)
)

# Do not assign car scores when no prior constructor history exists
constructor.loc[
    constructor["constructor_has_prior_history"] == 0,
    "car_performance_score"
] = np.nan

constructor["car_performance_tier"] = pd.qcut(
    constructor["car_performance_score"],
    q=5,
    labels=["Bottom", "Lower", "Midfield", "Upper", "Elite"]
)

# Keep score components for explainability
for col in score_components.columns:
    constructor[f"car_score_{col}"] = score_components[col]

constructor[[
    "season", "round", "constructor_name",
    "constructor_has_prior_history",
    "car_performance_score",
    "car_performance_tier"
]].sort_values(["season", "round", "car_performance_score"], ascending=[True, True, False]).head(30)

,season,round,constructor_name,constructor_has_prior_history,car_performance_score,car_performance_tier
471,2010,1,Ferrari,0,NaN,NaN
800,2010,1,Force India,0,NaN,NaN
1191,2010,1,HRT,0,NaN,NaN
1324,2010,1,Lotus,0,NaN,NaN
1456,2010,1,McLaren,0,NaN,NaN
1785,2010,1,Mercedes,0,NaN,NaN
2200,2010,1,Red Bull,0,NaN,NaN
2529,2010,1,Renault,0,NaN,NaN
2667,2010,1,Sauber,0,NaN,NaN
2892,2010,1,Toro Rosso,0,NaN,NaN


### Merge constructor features back to driver-race dataset

In [7]:
constructor_feature_cols = [
    "season",
    "round",
    "constructor_id",
    "constructor_race_id",
    "season_total_rounds",
    "season_progress",
    "season_phase",
    "regulation_era",
    "constructor_entry_count",
    "single_car_entry",
    "constructor_has_prior_history",
    "car_performance_score_raw",
    "car_performance_score",
    "car_performance_tier"
]

constructor_feature_cols += [
    col for col in constructor.columns
    if col.endswith("_expanding_prior")
    or col.endswith("_last5_prior")
    or col.endswith("_last3_prior")
    or col.endswith("_trend_last3_vs_expanding")
    or col.endswith("_momentum_last3_vs_last5")
    or col.startswith("car_score_")
]

constructor_feature_cols = list(dict.fromkeys(constructor_feature_cols))

missing_cols = [
    col for col in constructor_feature_cols
    if col not in constructor.columns
]

if missing_cols:
    raise KeyError(f"Missing constructor feature columns: {missing_cols}")

driver_race_with_constructor = race.merge(
    constructor[constructor_feature_cols],
    on=["season", "round", "constructor_id", "constructor_race_id"],
    how="left"
)

print("Driver-race input:", race.shape)
print("Driver-race with constructor features:", driver_race_with_constructor.shape)
print("Duplicate driver-race rows:",
      driver_race_with_constructor.duplicated(["season", "round", "driver_id"]).sum())

driver_race_with_constructor.head()

Driver-race input: (6911, 50)
Driver-race with constructor features: (6911, 111)
Duplicate driver-race rows: 0


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,constructor_qualifying_trend_last3_vs_expanding,constructor_points_trend_last3_vs_expanding,constructor_finish_momentum_last3_vs_last5,constructor_points_momentum_last3_vs_last5,car_score_finish_component,car_score_grid_component,car_score_qualifying_component,car_score_points_component,car_score_reliability_component,car_score_mechanical_component
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,liuzzi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,sutil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,bruno_senna,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Technical validation

In [8]:
print("=" * 80)
print("VALIDATION — CONSTRUCTOR FEATURES")
print("=" * 80)

print("\nDATASET SHAPES")
print("-" * 80)
print("Constructor features rows:", len(constructor))
print("Constructor features columns:", constructor.shape[1])
print("Driver-race enriched rows:", len(driver_race_with_constructor))
print("Driver-race enriched columns:", driver_race_with_constructor.shape[1])

print("\nROW INTEGRITY")
print("-" * 80)
constructor_dupes = constructor.duplicated(["season", "round", "constructor_id"]).sum()
driver_dupes = driver_race_with_constructor.duplicated(["season", "round", "driver_id"]).sum()

print("Duplicate constructor-race rows:", constructor_dupes)
print("Duplicate driver-race rows after merge:", driver_dupes)
print("Driver-race row count preserved:", len(driver_race_with_constructor) == len(race))

print("\nPRIOR HISTORY")
print("-" * 80)
print(constructor["constructor_has_prior_history"].value_counts())

print("\nCAR SCORE")
print("-" * 80)
print("Missing car scores:", constructor["car_performance_score"].isna().sum())
print("Min car score:", constructor["car_performance_score"].min())
print("Max car score:", constructor["car_performance_score"].max())

print("\nNO-HISTORY CAR SCORE CHECK")
print("-" * 80)
bad_no_history_scores = constructor[
    (constructor["constructor_has_prior_history"] == 0)
    & (constructor["car_performance_score"].notna())
]

print("No-history rows with car score:", len(bad_no_history_scores))

print("\nFIRST-RACE PRIOR FEATURE CHECK")
print("-" * 80)
first_constructor_races = (
    constructor
    .sort_values(["constructor_id", "season", "round"])
    .groupby("constructor_id")
    .head(1)
)

print("First constructor races:", len(first_constructor_races))
print("First constructor races with prior history:",
      first_constructor_races["constructor_has_prior_history"].sum())

issues = []

if constructor_dupes != 0:
    issues.append("Duplicate constructor-race rows exist.")

if driver_dupes != 0:
    issues.append("Duplicate driver-race rows exist after merge.")

if len(driver_race_with_constructor) != len(race):
    issues.append("Driver-race row count changed after merge.")

if len(bad_no_history_scores) != 0:
    issues.append("Rows without prior constructor history have car scores.")

if constructor["car_performance_score"].dropna().lt(0).any():
    issues.append("Car performance score below 0.")

if constructor["car_performance_score"].dropna().gt(100).any():
    issues.append("Car performance score above 100.")

if first_constructor_races["constructor_has_prior_history"].sum() != 0:
    issues.append("First constructor races should not have prior history.")

if issues:
    print("\nVERDICT: REVIEW NEEDED")
    for issue in issues:
        print("-", issue)
else:
    print("\nVERDICT: PASS — constructor features are technically valid.")

VALIDATION — CONSTRUCTOR FEATURES

DATASET SHAPES
--------------------------------------------------------------------------------
Constructor features rows: 3457
Constructor features columns: 80
Driver-race enriched rows: 6911
Driver-race enriched columns: 111

ROW INTEGRITY
--------------------------------------------------------------------------------
Duplicate constructor-race rows: 0
Duplicate driver-race rows after merge: 0
Driver-race row count preserved: True

PRIOR HISTORY
--------------------------------------------------------------------------------
constructor_has_prior_history
1    3434
0      23
Name: count, dtype: int64

CAR SCORE
--------------------------------------------------------------------------------
Missing car scores: 23
Min car score: 0.0
Max car score: 100.0

NO-HISTORY CAR SCORE CHECK
--------------------------------------------------------------------------------
No-history rows with car score: 0

FIRST-RACE PRIOR FEATURE CHECK
-------------------------

### Analytical supercheck

In [9]:
print("=" * 80)
print("SUPERCHECK — CONSTRUCTOR FEATURES")
print("=" * 80)

print("\n1. CAR SCORE DISTRIBUTION")
print("-" * 80)
print(constructor["car_performance_score"].describe())
print("\nTier distribution:")
print(constructor["car_performance_tier"].value_counts(dropna=False))

print("\n2. BEST CONSTRUCTOR-SEASONS")
print("-" * 80)
constructor_season_scores = (
    constructor
    .dropna(subset=["car_performance_score"])
    .groupby(["season", "constructor_name"], as_index=False)
    .agg(
        avg_car_score=("car_performance_score", "mean"),
        min_car_score=("car_performance_score", "min"),
        max_car_score=("car_performance_score", "max"),
        races=("round", "nunique")
    )
)

display(
    constructor_season_scores
    .sort_values("avg_car_score", ascending=False)
    .head(25)
)

print("\n3. WORST CONSTRUCTOR-SEASONS")
print("-" * 80)
display(
    constructor_season_scores
    .sort_values("avg_car_score", ascending=True)
    .head(25)
)

print("\n4. KNOWN HISTORICAL SANITY CHECKS")
print("-" * 80)

known_checks = [
    (2010, "Red Bull"),
    (2011, "Red Bull"),
    (2013, "Red Bull"),
    (2014, "Mercedes"),
    (2016, "Mercedes"),
    (2020, "Mercedes"),
    (2021, "Red Bull"),
    (2022, "Red Bull"),
    (2023, "Red Bull"),
    (2024, "McLaren"),
    (2024, "Red Bull"),
    (2024, "Ferrari")
]

known_rows = []

for season, constructor_name in known_checks:
    temp = constructor[
        (constructor["season"] == season)
        & (constructor["constructor_name"] == constructor_name)
    ].dropna(subset=["car_performance_score"])

    if len(temp) > 0:
        known_rows.append({
            "season": season,
            "constructor_name": constructor_name,
            "avg_car_score": temp["car_performance_score"].mean(),
            "min_car_score": temp["car_performance_score"].min(),
            "max_car_score": temp["car_performance_score"].max(),
            "races_with_score": temp["round"].nunique()
        })

known_df = pd.DataFrame(known_rows)
display(known_df.sort_values(["season", "avg_car_score"], ascending=[True, False]))

print("\n5. DEVELOPMENT PATH EXAMPLES")
print("-" * 80)

selected_paths = [
    (2014, "Mercedes"),
    (2021, "Red Bull"),
    (2023, "Aston Martin"),
    (2023, "McLaren"),
    (2024, "McLaren"),
    (2024, "Ferrari"),
    (2024, "Red Bull")
]

path_frames = []

for season, constructor_name in selected_paths:
    temp = (
        constructor[
            (constructor["season"] == season)
            & (constructor["constructor_name"] == constructor_name)
        ][[
            "season",
            "round",
            "constructor_name",
            "season_progress",
            "car_performance_score",
            "constructor_finish_trend_last3_vs_expanding",
            "constructor_points_trend_last3_vs_expanding",
            "constructor_finish_momentum_last3_vs_last5",
            "constructor_points_momentum_last3_vs_last5"
        ]]
        .sort_values(["season", "round"])
    )

    if len(temp) > 0:
        path_frames.append(temp)

if path_frames:
    display(pd.concat(path_frames, ignore_index=True))
else:
    print("No selected path examples found.")

print("\n6. CAR SCORE CORRELATION CHECK")
print("-" * 80)

corr_cols = [
    "car_performance_score",
    "constructor_avg_finish_last5_prior",
    "constructor_avg_grid_last5_prior",
    "constructor_avg_qualifying_last5_prior",
    "constructor_total_points_last5_prior",
    "constructor_finished_rate_last5_prior",
    "constructor_mechanical_dnf_rate_last5_prior",
    "constructor_finish_std_last5_prior",
    "constructor_finish_trend_last3_vs_expanding",
    "constructor_points_trend_last3_vs_expanding"
]

corr = constructor[corr_cols].corr(numeric_only=True)
display(corr)

print("\nCorrelation with car_performance_score:")
print(corr["car_performance_score"].sort_values(ascending=False))

print("\n7. SUPERCHECK VERDICT")
print("-" * 80)

supercheck_issues = []

if constructor["car_performance_score"].dropna().min() < 0:
    supercheck_issues.append("Car score below 0.")

if constructor["car_performance_score"].dropna().max() > 100:
    supercheck_issues.append("Car score above 100.")

if constructor_season_scores.empty:
    supercheck_issues.append("Constructor season scores are empty.")

if supercheck_issues:
    print("REVIEW NEEDED")
    for issue in supercheck_issues:
        print("-", issue)
else:
    print("PASS — constructor features look analytically reasonable.")

SUPERCHECK — CONSTRUCTOR FEATURES

1. CAR SCORE DISTRIBUTION
--------------------------------------------------------------------------------
count    3434.000000
mean       65.881482
std        14.999499
min         0.000000
25%        55.788131
50%        64.522387
75%        77.032911
max       100.000000
Name: car_performance_score, dtype: float64

Tier distribution:
car_performance_tier
Bottom      687
Lower       687
Upper       687
Elite       687
Midfield    686
NaN          23
Name: count, dtype: int64

2. BEST CONSTRUCTOR-SEASONS
--------------------------------------------------------------------------------


,season,constructor_name,avg_car_score,min_car_score,max_car_score,races
103,2019,Mercedes,94.710599,88.099589,99.622595,21
63,2015,Mercedes,94.281217,81.340584,99.755033,19
83,2017,Mercedes,93.617932,86.412511,99.755033,20
18,2011,Red Bull,92.652995,84.443112,97.686367,19
73,2016,Mercedes,92.376980,83.837108,100.000000,21
114,2020,Mercedes,92.298154,86.949254,97.411990,17
163,2025,McLaren,90.906522,83.464693,96.699561,24
147,2023,Red Bull,90.295050,79.913080,95.131364,22
93,2018,Mercedes,89.626701,79.255668,96.946318,21
153,2024,McLaren,87.236553,72.140048,94.946363,24



3. WORST CONSTRUCTOR-SEASONS
--------------------------------------------------------------------------------


,season,constructor_name,avg_car_score,min_car_score,max_car_score,races
10,2010,Virgin,28.276607,0.000000,42.495441,18
2,2010,HRT,29.826924,10.396240,41.767221,18
3,2010,Lotus,33.087617,20.993648,45.115173,18
27,2012,HRT,37.330108,30.704799,41.355596,19
47,2014,Caterham,38.001591,32.799700,46.675503,17
8,2010,Sauber,41.023065,7.858952,63.438569,18
15,2011,Lotus,41.298067,27.127775,50.583643,19
22,2011,Virgin,42.412545,32.105817,51.337693,19
14,2011,HRT,42.663370,35.990300,48.366670,18
29,2012,Marussia,45.044449,32.066246,54.218716,19



4. KNOWN HISTORICAL SANITY CHECKS
--------------------------------------------------------------------------------


...


5. DEVELOPMENT PATH EXAMPLES
--------------------------------------------------------------------------------


,season,round,constructor_name,season_progress,car_performance_score,constructor_finish_trend_last3_vs_expanding,constructor_points_trend_last3_vs_expanding,constructor_finish_momentum_last3_vs_last5,constructor_points_momentum_last3_vs_last5
0,2014,1,Mercedes,0.052632,81.891147,3.580087,4.225108,1.133333,0.266667
1,2014,2,Mercedes,0.105263,81.476308,1.916667,5.384615,-1.333333,-2.600000
2,2014,3,Mercedes,0.157895,82.939043,3.478903,14.654008,-0.166667,3.666667
3,2014,4,Mercedes,0.210526,85.203476,5.210417,24.600000,0.966667,9.600000
4,2014,5,Mercedes,0.263158,89.192973,7.944444,30.222222,2.800000,9.800000
...,...,...,...,...,...,...,...,...,...
152,2024,20,Red Bull,0.833333,81.492161,-1.220000,-7.795000,-0.733333,-0.933333
153,2024,21,Red Bull,0.875000,77.768223,-1.368771,-8.406977,0.600000,2.000000
154,2024,22,Red Bull,0.916667,77.235685,-1.369205,-6.078918,0.400000,1.533333
155,2024,23,Red Bull,0.958333,78.666744,-2.198020,-9.367987,-1.133333,-2.000000



6. CAR SCORE CORRELATION CHECK
--------------------------------------------------------------------------------


,car_performance_score,constructor_avg_finish_last5_prior,constructor_avg_grid_last5_prior,constructor_avg_qualifying_last5_prior,constructor_total_points_last5_prior,constructor_finished_rate_last5_prior,constructor_mechanical_dnf_rate_last5_prior,constructor_finish_std_last5_prior,constructor_finish_trend_last3_vs_expanding,constructor_points_trend_last3_vs_expanding
car_performance_score,1.000000,-0.951188,-0.890255,-0.887436,0.871064,0.654965,-0.566984,0.110706,0.448348,0.405130
constructor_avg_finish_last5_prior,-0.951188,1.000000,0.910662,0.914396,-0.909475,-0.503491,0.348502,-0.159976,-0.469314,-0.406130
constructor_avg_grid_last5_prior,-0.890255,0.910662,1.000000,0.985656,-0.851120,-0.310331,0.234943,-0.313855,-0.327740,-0.333525
constructor_avg_qualifying_last5_prior,-0.887436,0.914396,0.985656,1.000000,-0.855246,-0.295441,0.228380,-0.332723,-0.322987,-0.322213
constructor_total_points_last5_prior,0.871064,-0.909475,-0.851120,-0.855246,1.000000,0.355564,-0.228769,0.150117,0.418713,0.505764
constructor_finished_rate_last5_prior,0.654965,-0.503491,-0.310331,-0.295441,0.355564,1.000000,-0.693343,-0.308725,0.398871,0.241771
constructor_mechanical_dnf_rate_last5_prior,-0.566984,0.348502,0.234943,0.228380,-0.228769,-0.693343,1.000000,0.114132,-0.224260,-0.144090
constructor_finish_std_last5_prior,0.110706,-0.159976,-0.313855,-0.332723,0.150117,-0.308725,0.114132,1.000000,-0.146656,-0.095480
constructor_finish_trend_last3_vs_expanding,0.448348,-0.469314,-0.327740,-0.322987,0.418713,0.398871,-0.224260,-0.146656,1.000000,0.820001
constructor_points_trend_last3_vs_expanding,0.405130,-0.406130,-0.333525,-0.322213,0.505764,0.241771,-0.144090,-0.095480,0.820001,1.000000



Correlation with car_performance_score:
car_performance_score                          1.000000
constructor_total_points_last5_prior           0.871064
constructor_finished_rate_last5_prior          0.654965
constructor_finish_trend_last3_vs_expanding    0.448348
constructor_points_trend_last3_vs_expanding    0.405130
constructor_finish_std_last5_prior             0.110706
constructor_mechanical_dnf_rate_last5_prior   -0.566984
constructor_avg_qualifying_last5_prior        -0.887436
constructor_avg_grid_last5_prior              -0.890255
constructor_avg_finish_last5_prior            -0.951188
Name: car_performance_score, dtype: float64

7. SUPERCHECK VERDICT
--------------------------------------------------------------------------------
PASS — constructor features look analytically reasonable.


### Save outputs

In [10]:
constructor_features_path = PROCESSED / "constructor_features.csv"
driver_enriched_path = PROCESSED / "driver_race_with_constructor_features.csv"

constructor.to_csv(constructor_features_path, index=False)
driver_race_with_constructor.to_csv(driver_enriched_path, index=False)

constructor_saved = pd.read_csv(constructor_features_path)
driver_saved = pd.read_csv(driver_enriched_path)

print("Saved constructor features:", constructor_features_path.exists())
print("Constructor rows match:", len(constructor_saved) == len(constructor))
print("Constructor columns match:", constructor_saved.shape[1] == constructor.shape[1])

print("\nSaved driver-race enriched dataset:", driver_enriched_path.exists())
print("Driver rows match:", len(driver_saved) == len(driver_race_with_constructor))
print("Driver columns match:", driver_saved.shape[1] == driver_race_with_constructor.shape[1])

Saved constructor features: True
Constructor rows match: True
Constructor columns match: True

Saved driver-race enriched dataset: True
Driver rows match: True
Driver columns match: True
